In [1]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [5]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [6]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [7]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Control Packet Hierachy

### CQ1.01g - What types of COntrol Packets exists ?

In [8]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX mqv: <https://www.w3.org/2019/wot/mqtt#> 

SELECT ?type WHERE {
  ?type rdfs:subClassOf mqtt:ControlPacket .

    FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,type
0,mqtt:AuthPacket
1,mqtt:ConnackPacket
2,mqtt:ConnectPacket
3,mqtt:DisconnectPacket
4,mqtt:ControlPacket
5,mqtt:PingreqPacket
6,mqtt:PingrespPacket
7,mqtt:PubackPacket
8,mqtt:PubcompPacket
9,mqtt:PublishPacket


### CQ1.02g - Which Control Packets has a fixed / Variable Header or Payload ?

In [18]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 


SELECT ?type ?relation ?entity WHERE {
  ?type rdfs:subClassOf mqtt:ControlPacket .
  ?relation rdfs:range ?type ;
       rdfs:domain ?entity .
  FILTER( STRSTARTS(STR(?relation), STR(mqtt:)) ) .
  FILTER( STRSTARTS(STR(?entity), STR(mqtt:)) )

}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,type,relation,entity
0,mqtt:AuthPacket,mqtt:isAuthVariableHeaderOf,mqtt:AuthVariableHeader
1,mqtt:AuthPacket,mqtt:isAuthFixedHeaderOf,mqtt:AuthFixedHeader
2,mqtt:ConnackPacket,mqtt:isConnackVariableHeaderOf,mqtt:ConnackVariableHeader
3,mqtt:ConnackPacket,mqtt:isConnackFixedHeaderOf,mqtt:ConnackFixedHeader
4,mqtt:ConnackPacket,mqtt:receivesConnackPacket,mqtt:Client
5,mqtt:ConnackPacket,mqtt:sendsConnackPacket,mqtt:Broker
6,mqtt:ConnectPacket,mqtt:isConnectFixedHeaderOf,mqtt:ConnectFixedHeader
7,mqtt:ConnectPacket,mqtt:isConnectPayloadOf,mqtt:ConnectPayload
8,mqtt:ConnectPacket,mqtt:isConnectVariableHeaderOf,mqtt:ConnectVariableHeader
9,mqtt:ConnectPacket,mqtt:receivesConnectPacket,mqtt:Broker
